In [ ]:
import cv2
import datetime
import numpy as np

from skimage.exposure import rescale_intensity

from breathecam import BreatheCam
from bbox import *
from clhyde import *
from utilities import *

In [ ]:
import matplotlib.animation as animation
import matplotlib.pyplot as plt

from IPython.display import HTML

def animate(source, single_frame=False, color_map=None):
    """Displays a single image or animates a list of images

    Parameters
    ---
    * `source` - the frames to animate
    * `single_frame` - whether `source` is an image
    * `color_map` - color map to use when displaying images

    """
    fig, ax = plt.subplots(1, 1)
    fig.tight_layout()


    if single_frame:
        ax.imshow(source, cmap=color_map)
    else:
        ims = [[ax.imshow(f, cmap=color_map, animated=True)] for f in source]
        anim = animation.ArtistAnimation(fig, ims, interval=167, blit=True, repeat_delay=1000)

        plt.close()
        display(HTML(anim.to_jshtml()))

In [ ]:
day = datetime.date.fromisoformat("2024-05-19")
time = datetime.time.fromisoformat("09:50:00")
nframes = 41
level = 4
bbox = BoundingBox(3000, 1500, 6500, 2500)
camera = BreatheCam("Clairton Coke Works", day)
video_rgb_u8 = camera.extract(time, nframes, bbox=bbox, seconds_per_frame=3, level=level)
video_gray_u8 = np.array([cv2.cvtColor(f, cv2.COLOR_RGB2GRAY) for f in video_rgb_u8])

Rescaling the intensity helps to enhance less opaque smoke

In [ ]:
intensity_minimum = 25
intensity_maximum = 200

video_rescaled_u8 = np.array([rescale_intensity(f, in_range=(intensity_minimum, intensity_maximum)) for f in video_gray_u8])
video_rescaled_f32 = video_rescaled_u8.astype(np.float32) / 255.0
# animate(rescaled_f32)

## LSBP Background Subtractor

### Parameters

|          Parameter           |                Default                 | Description                                                         |
|:----------------------------:|:--------------------------------------:|:--------------------------------------------------------------------|
|             `mc`             | `LBSP_CAMERA_MOTION_COMPENSATION_NONE` | Whether to use camera motion compensation                           |
|          `nSamples`          |                  `20`                  | Number of samples to maintain at each point of the frame            |
|         `LSBPRadius`         |                  `16`                  | LSBP descriptor radius                                              |
|           `Tlower`           |                 `2.0`                  | Lower bound for T-values                                            |
|           `Tupper`           |                 `32.0`                 | Upper bound for T-values                                            |
|            `Tinc`            |                 `1.0`                  | Increase step for T-values                                          |
|            `Tdec`            |                 `0.05`                 | Decrease step for T-values                                          |
|           `Rscale`           |                 `10.0`                 | Scale coefficient for threshold values                              |
|          `Rincdec`           |                `0.005`                 | Increase/Decrease step for threshold values                         |
| `noiseRemovalThresholdFacBG` |                `0.0004`                | Strength of the noise removal for background points                 |
| `noiseRemovalThresholdFacFG` |                `0.0008`                | Strength of the noise removal for foreground points                 |
|       `LSBPthreshold`        |                  `8`                   | Threshold for LSBP binary string                                    |
|          `minCount`          |                  `2`                   | Minimal number of matches for sample to be considered as foreground |


In [ ]:
bgs = cv2.bgsegm.createBackgroundSubtractorLSBP(
    mc=cv2.bgsegm.LSBP_CAMERA_MOTION_COMPENSATION_NONE,
    nSamples=20,
    LSBPRadius=64,
    Tlower=2.0,
    Tupper=32.0,
    Tinc=1.0,
    Tdec=0.05,
    Rscale=10.0,
    Rincdec=0.005,
    noiseRemovalThresholdFacBG=0.0004,
    noiseRemovalThresholdFacFG=0.0008,
    LSBPthreshold=32,
    minCount=2
)

In [ ]:
foreground_f32 = np.array([bgs.apply(f) for f in video_rescaled_f32])

## Clustering

Clustering is based on color difference of nearest-neighbors and past behavior. These neighbors are chosen based on an array of offsets from the current pixel. The default below uses the 8 surrounding neighbors and at most 5 frames from the past.

In [ ]:
NEIGHBOR_OFFSETS = np.array([
    (0, 0, -1), (0, -1, -1), (0, -1,  0), (0, -1,  1),
    (0, 0,  1), (0,  1,  1), (0,  1,  0), (0,  1,  1),
    (-1, 0, 0), (-2, 0, 0), (-3, 0, 0), (-4, 0, 0),
    (-5, 0, 0)
])

We choose the CIELAB colorspace and the $\Delta E_{HyAB}$ formula for determining color difference. $$\Delta E_{HyAB} = |L_2 - L_1| + \sqrt{(a_2 - a_1)^2 + (b_2 - b_1)^2}$$

In [ ]:
video_cielab_f32 = np.array([cv2.cvtColor(f, cv2.COLOR_RGB2LAB) for f in video_rgb_u8.astype(np.float32) / 255.0])

Note that `cluster` returns a generator.

In [ ]:
temporal_contours = list(cluster(
    cielab_video=video_cielab_f32,
    mask=foreground_f32,
    neighbor_offsets=NEIGHBOR_OFFSETS,
    threshold=20
))

temporal_contours_large = [c for c in temporal_contours if c.height > 8 and c.width > 8]

## Visualizing Contours

For each contour we pick a random color and label all the pixels in the contour with that color.

`contour.mask` and `contour.mask_from` can be used to generate the mask corresponding to any individual contour.

In [ ]:
visual = np.zeros_like(video_rgb_u8, dtype=np.uint8)

for contour in temporal_contours_large:
    color = tuple(int(c) for c in np.random.randint(64, 192, size=(3,)))

    for point in contour.points:
        visual[*point] = color

animate(visual)